In [ ]:
import pandas as pd
import numpy as np
import ast
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# --- 1. CLEANING HELPER FUNCTIONS ---

def filter_redundant_ingredients(ing_list):
    """
    Advanced filter: If 'basmati rice' words are entirely contained
    within 'basmati brown rice', the shorter one is removed.
    """
    # Remove exact duplicates and sort by length (longest first)
    unique_set = list(set(ing_list))
    sorted_ings = sorted(unique_set, key=len, reverse=True)

    kept_ings = []
    for current_ing in sorted_ings:
        current_tokens = set(current_ing.split())

        is_redundant = False
        for kept_ing in kept_ings:
            kept_tokens = set(kept_ing.split())
            # Check if all words of current item exist in a longer item already kept
            if current_tokens.issubset(kept_tokens):
                is_redundant = True
                break

        if not is_redundant:
            kept_ings.append(current_ing)

    return kept_ings

def clean_text_field(text, url_pattern):
    """Removes URLs and cleans whitespace from strings."""
    if not isinstance(text, str):
        return ""
    return re.sub(url_pattern, '', text).strip()

# --- 2. THE CLEANING PIPELINE ---

def load_and_clean_dataset(filepath):
    print("Step 1: Loading Data...")
    df = pd.read_csv(filepath)

    # Configuration
    dropped_ings = {"salt", "pepper", "oil", "butter", "margarine", "water"}
    url_pattern = r'http\S+|www\.\S+'

    print("Step 2: Processing NER lists and removing redundancies...")
    # Convert string-lists to Python lists
    df['NER_list'] = df['NER'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

    # Clean ingredients: lowercase, remove URLs, remove common items
    df['NER_list'] = df['NER_list'].apply(lambda x: [
        re.sub(url_pattern, '', ing).strip().lower()
        for ing in x if ing.lower() not in dropped_ings
    ])

    # Apply the smart Token-Subset filter (Fixes the Basmati issue)
    df['NER_list'] = df['NER_list'].apply(filter_redundant_ingredients)

    print("Step 3: Cleaning text columns and dropping empty rows...")
    for col in ['title', 'ingredients', 'directions']:
        df[col] = df[col].apply(lambda x: clean_text_field(x, url_pattern))

    # Remove rows with missing essential data or too few ingredients
    df = df.dropna(subset=['title', 'ingredients', 'directions'])
    df = df[(df['title'] != "") & (df['NER_list'].map(len) > 3)]

    # Step 4: Drop duplicate recipes (exact same ingredient set)
    df['temp_key'] = df['NER_list'].apply(lambda x: tuple(sorted(x)))
    df = df.drop_duplicates(subset=['temp_key']).drop(columns=['temp_key'])

    df = df.reset_index(drop=True)
    print(f"Success! Processed {len(df)} unique recipes.")
    return df

# --- 3. MODEL BUILDING ---

# Run pipeline
data = load_and_clean_dataset("raw_data/recipes_data.csv")

print("Step 4: Vectorizing data...")
# Join cleaned ingredients into a single string for TF-IDF
corpus = data['NER_list'].apply(lambda x: " ".join(x))
vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(corpus)

# --- 4. RECOMMENDATION SYSTEM ---

def get_recommendations(user_query, top_n=5):
    # Standardize the query
    query_vec = vectorizer.transform([user_query.lower().strip()])

    # Calculate Cosine Similarity
    similarities = cosine_similarity(query_vec, tfidf_matrix).flatten()

    # Get top indices
    related_indices = similarities.argsort()[-top_n:][::-1]

    print(f"\nTop Results for: '{user_query}'")
    print("=" * 40)
    for i in related_indices:
        row = data.iloc[i]
        print(f"Recipe: {row['title']} (Score: {similarities[i]:.4f})")
        print(f"Ingredients: {row['NER_list']}\n")

# --- TEST ---
get_recommendations("rice basmati")

Step 1: Loading Data...
Step 2: Processing NER lists and removing redundancies...
Step 3: Cleaning text columns and dropping empty rows...
Success! Processed 1778249 unique recipes.
Step 4: Vectorizing data...

Top Results for: 'rice basmati'
Recipe: Lite Rice Pudding (Score: 0.8239)
Ingredients: ['basmati rice', 'cinnamon', 'vanilla', 'sugar', 'milk']

Recipe: Parslied Rice With Browned Garlic Butter (Score: 0.8174)
Ingredients: ['basmati rice', 'red pepper', 'parsley', 'garlic', '¼']

Recipe: Artichoke Pilaf (Score: 0.8027)
Ingredients: ['basmati rice', 'olive oil', 'garlic', 'thyme']

Recipe: Apple Fried Rice (Score: 0.7874)
Ingredients: ['pepper powder', 'basmati rice', 'apple', 'sugar']

Recipe: Vanilla Rice (Score: 0.7787)
Ingredients: ['basmati rice', 'canola oil', 'vanilla', 'sugar']

